In [2]:
#import liberies
import ee
import geemap
import pandas as pd
import geopandas as gpd

In [3]:
ee.Authenticate()

True

In [4]:
import ee
ee.Initialize()

## DEM AND COMPUTE SLOPE

In [5]:

# Load the SRTM Digital Elevation Model (DEM)
dem = ee.Image("USGS/SRTMGL1_003")

# Compute slope from the DEM
slope = ee.Terrain.slope(dem)

# Visualization parameters
dem_vis = {
    "min": 0,
    "max": 3000,
    "palette": [
        "006633",  # Dark Green
        "66cc66",  # Green
        "ffff99",  # Yellow
        "cc9966",  # Brown
        "ffffff"   # White
    ]
}

slope_vis = {
    "min": 0,
    "max": 60,
    "palette": [
        "ffffff",  # White
        "ffff00",  # Yellow
        "ff9900",  # Orange
        "ff0000"   # Red
    ]
}

# Create the map
Map = geemap.Map()

# Center the map on Nigeria
Map.setCenter(8.6753, 9.0820, 6)

# Add DEM and Slope layers
Map.addLayer(dem, dem_vis, "SRTM DEM")
Map.addLayer(slope, slope_vis, "Slope")

# Layer control
Map.addLayerControl()

# Display the map
Map

Map(center=[9.082, 8.6753], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## ESA Worldcover from GEE

In [6]:
# Load the ESA WorldCover dataset (2021)
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

# Visualization parameters
vis_params = {
    "min": 10,
    "max": 100,
    "palette": [
        "#006400",  # 10 Tree cover
        "#ffbb22",  # 20 Shrubland
        "#ffff4c",  # 30 Grassland
        "#f096ff",  # 40 Cropland
        "#fa0000",  # 50 Built-up
        "#b4b4b4",  # 60 Bare / Sparse vegetation
        "#f0f0f0",  # 70 Snow and Ice
        "#0064c8",  # 80 Permanent Water Bodies
        "#0096a0",  # 90 Herbaceous Wetland
        "#00cf75",  # 95 Mangroves
        "#fae6a0"   # 100 Moss and Lichen
    ]
}

# Create the map
Map = geemap.Map()

# Center on Nigeria
Map.setCenter(8.6753, 9.0820, 6)

# Add the WorldCover layer
Map.addLayer(
    worldcover,
    vis_params,
    "ESA WorldCover 2021"
)

# Add layer control
Map.addLayerControl()

# Display the map
Map

Map(center=[9.082, 8.6753], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [7]:
legend_dict = {
    "Tree Cover": "006400",
    "Shrubland": "ffbb22",
    "Grassland": "ffff4c",
    "Cropland": "f096ff",
    "Built-up": "fa0000",
    "Bare / Sparse Vegetation": "b4b4b4",
    "Snow and Ice": "f0f0f0",
    "Permanent Water": "0064c8",
    "Herbaceous Wetland": "0096a0",
    "Mangroves": "00cf75",
    "Moss and Lichen": "fae6a0",
}

Map.add_legend(
    title="ESA WorldCover",
    legend_dict=legend_dict
)

Map

Map(bottom=8077.0, center=[9.082, 8.6753], controls=(WidgetControl(options=['position', 'transparent_bg'], pos…

In [8]:
legend = {
    "Sand": "d5c36b",
    "Loamy Sand": "b96947",
    "Sandy Loam": "9d3706",
    "Loam": "ae868f",
    "Silt Loam": "f86714",
    "Silt": "46d143",
    "Sandy Clay Loam": "368f20",
    "Clay Loam": "3e5a14",
    "Silty Clay Loam": "ffd557",
    "Sandy Clay": "fff72e",
    "Silty Clay": "ff5a9d",
    "Clay": "ff005b"
}

Map.add_legend(
    title="USDA Soil Texture",
    legend_dict=legend
)

Map

Map(bottom=8077.0, center=[9.082, 8.6753], controls=(WidgetControl(options=['position', 'transparent_bg'], pos…

## Compute Drainage Density using GEE

In [9]:
# Load administrative boundaries and filter to Kano State, Nigeria
admin1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

kano = admin1.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Nigeria"),
        ee.Filter.eq("ADM1_NAME", "Kano")
    )
)

aoi = kano.geometry()

## SOIL TEXTURE WITH GEE

In [10]:


# Load OpenLandMap Soil Texture dataset
soil_texture = ee.Image(
    "OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02"
).select("b0").clip(aoi)
# Select the topsoil texture layer (0–5 cm)
soil = soil_texture.select("b0")

# Visualization parameters
vis_params = {
    "min": 1,
    "max": 12,
    "palette": [
        "#d5c36b",  # Sand
        "#b96947",  # Loamy Sand
        "#9d3706",  # Sandy Loam
        "#ae868f",  # Loam
        "#f86714",  # Silt Loam
        "#46d143",  # Silt
        "#368f20",  # Sandy Clay Loam
        "#3e5a14",  # Clay Loam
        "#ffd557",  # Silty Clay Loam
        "#fff72e",  # Sandy Clay
        "#ff5a9d",  # Silty Clay
        "#ff005b"   # Clay
    ]
}

# Create map
Map = geemap.Map()

# Center on Nigeria
Map.setCenter(8.6753, 9.0820, 6)

# Add soil texture layer
Map.addLayer(
    soil,
    vis_params,
    "Soil Texture (0–5 cm)"
)

# Add layer control
Map.addLayerControl()

# Display map
Map

Map(center=[9.082, 8.6753], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [11]:
# Load the FAO GAUL Level 1 administrative boundaries
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Filter the dataset to only Kano State
kano = gaul.filter(ee.Filter.eq("ADM1_NAME", "Kano"))

# Create a new map
Map = geemap.Map()

# Center the map on Kano
Map.centerObject(kano, 8)

# Add the Kano boundary to the map
Map.addLayer(
    kano,
    {"color": "red"},
    "Kano Boundary"
)

# Display the map
Map

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

## Compute the Topographic Wetness Index (TWI)

In [12]:
# Create map
Map = geemap.Map()

# ------------------------------------
# Area of Interest (replace with your AOI)
# ------------------------------------
aoi = ee.Geometry.Rectangle([8.3, 6.7, 9.5, 7.5])


# ------------------------------------
# MERIT Hydro
# ------------------------------------
merit = ee.Image("MERIT/Hydro/v1_0_1")

# Flow accumulation (upstream area)
flow_acc = merit.select("upa")

# ------------------------------------
# DEM
# ------------------------------------
dem = ee.Image("MERIT/DEM/v1_0_3")

# ------------------------------------
# Compute slope
# ------------------------------------
slope = ee.Terrain.slope(dem)


slope_rad = slope.multiply(3.14159265 / 180)




tan_slope = slope_rad.tan().max(0.001)

twi = flow_acc.divide(tan_slope).log()

Map.centerObject(aoi, 9)

twi_vis = {
    "min": 0,
    "max": 12,
    "palette": [
        "white",
        "yellow",
        "green",
        "blue"
    ]
}

Map.addLayer(
    twi.clip(aoi),
    twi_vis,
    "Topographic Wetness Index"
)

Map

Map(center=[7.100269362384542, 8.90000000000003], controls=(WidgetControl(options=['position', 'transparent_bg…

In [13]:
#Visualization parameters
twi_vis = {
    "min": 0,
    "max": 12,
    "palette": [
        "white",
        "yellow",
        "green",
        "blue"
    ]
}

## Resample Thematic layers to 30m

In [14]:
aoi = ee.Geometry.Rectangle([8.3, 6.7, 9.1, 7.5])

# Elevation (MERIT DEM)
elevation = ee.Image("MERIT/DEM/v1_0_3").clip(aoi)

# Slope from DEM
slope = ee.Terrain.slope(elevation)

# Flow accumulation from MERIT Hydro
merit = ee.Image("MERIT/Hydro/v1_0_1")
flow_acc = merit.select("upa")

# Example drainage density (replace with your class method)
drainage_density = flow_acc.gt(100)

# Example TWI
slope_rad = slope.multiply(3.14159265 / 180)
twi = flow_acc.divide(slope_rad.tan().max(0.001)).log()

# Rainfall (example dataset)
rainfall = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
      .filterDate("2023-01-01", "2023-12-31")
      .sum()
      .clip(aoi)
)

# Land cover (example dataset)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(aoi)

# Soil texture
# Replace this with the dataset used in your class
soil_texture = ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02").clip(aoi)

## Export Thematic layers at 30m Spatial Resolution & Clipped to Kano Boundary

In [18]:
#load Nigeria state boundaries
states = ee.FeatureCollection("FAO/GAUL/2015/level1")

#filter the dataset to only Kano State
kano_boundary = states.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Nigeria"),
        ee.Filter.eq("ADM1_NAME", "Kano")

    )
)

Map = geemap.Map()
Map.centerObject(kano_boundary, 8)
Map.addLayer(kano_boundary, {"color": "red"}, "Kano Boundary")
Map

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

In [19]:
#Clip Each Resampled Layer to Kano Boundary
rainfall_clip = rainfall.clip(kano_boundary)
elevation_clip = elevation.clip(kano_boundary)
slope_clip = slope.clip(kano_boundary)
land_cover_clip = land_cover.clip(kano_boundary)
soil_texture_clip = soil_texture.clip(kano_boundary)
drainage_density_clip = drainage_density.clip(kano_boundary)
twi_clip = twi.clip(kano_boundary)


#Confirm Resolution is 30 m
rainfall_clip = rainfall_clip.reproject(crs='EPSG:4326', scale=30)
elevation_clip = elevation_clip.reproject(crs='EPSG:4326', scale=30)
slope_clip = slope_clip.reproject(crs='EPSG:4326', scale=30)
land_cover_clip = land_cover_clip.reproject(crs='EPSG:4326', scale=30)
soil_texture_clip = soil_texture_clip.reproject(crs='EPSG:4326', scale=30)
drainage_density_clip = drainage_density_clip.reproject(crs='EPSG:4326', scale=30)
twi_clip = twi_clip.reproject(crs='EPSG:4326', scale=30)

print(kano_boundary)

ee.FeatureCollection({
  "functionInvocationValue": {
    "functionName": "Collection.filter",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection.loadTable",
          "arguments": {
            "tableId": {
              "constantValue": "FAO/GAUL/2015/level1"
            }
          }
        }
      },
      "filter": {
        "functionInvocationValue": {
          "functionName": "Filter.and",
          "arguments": {
            "filters": {
              "arrayValue": {
                "values": [
                  {
                    "functionInvocationValue": {
                      "functionName": "Filter.equals",
                      "arguments": {
                        "leftField": {
                          "constantValue": "ADM0_NAME"
                        },
                        "rightValue": {
                          "constantValue": "Nigeria"
                        }
                      }


In [20]:
#Export function
def export_to_drive(image, description, folder):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        region=kano_boundary.geometry(),
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()


layers = {
    "rainfall_clip": rainfall_clip,
    "elevation_clip": elevation_clip,
    "slope_clip": slope_clip,
    "land_cover_clip": land_cover_clip,
    "soil_texture_clip": soil_texture_clip,
    "drainage_density_clip": drainage_density_clip,
    "twi_clip": twi_clip
}

for name, layer in layers.items():
    print(name, type(layer))
    




rainfall_clip <class 'ee.image.Image'>
elevation_clip <class 'ee.image.Image'>
slope_clip <class 'ee.image.Image'>
land_cover_clip <class 'ee.image.Image'>
soil_texture_clip <class 'ee.image.Image'>
drainage_density_clip <class 'ee.image.Image'>
twi_clip <class 'ee.image.Image'>


In [35]:
#cancel old READY task
for task in ee.batch.Task.list():
    status = task.status()
    if status['state'] == 'READY':
        task.cancel()
        print("Cancelled:", status['description'])

#Recreate clean exports
print(type(kano_boundary))
print(type(rainfall_clip))

<class 'ee.featurecollection.FeatureCollection'>
<class 'ee.image.Image'>


## Export all layers to Google Drive


In [ ]:
folder_name = "Kano_Thematic_30m"

export_to_drive(rainfall_clip, "rainfall_30m", folder_name)
export_to_drive(elevation_clip, "elevation_30m", folder_name)
export_to_drive(slope_clip, "slope_30m", folder_name)
export_to_drive(land_cover_clip, "land_cover_30m", folder_name)
export_to_drive(soil_texture_clip, "soil_texture_30m", folder_name)
export_to_drive(drainage_density_clip, "drainage_density_30m", folder_name)
export_to_drive(twi_clip, "twi_30m", folder_name)

## Check the status of the export tasks

In [36]:

for task in ee.batch.Task.list():
    status = task.status()
    print(status['description'], ":", status['state'])

Kano_Reclassified_Slope : RUNNING
Kano_Reclassified_Elevation : COMPLETED
Kano_Reclassified_Rainfall : COMPLETED
Kano_Reclassified_TWI : COMPLETED
Kano_Reclassified_Drainage : COMPLETED
Kano_Reclassified_Soil : COMPLETED
Kano_Reclassified_Landcover : COMPLETED
Kano_Reclassified_Slope : COMPLETED
Kano_Reclassified_Elevation : COMPLETED
Kano_Reclassified_Rainfall : COMPLETED
Kano_Land_Cover_30m : COMPLETED
Kano_TWI_30m : COMPLETED
Kano_Drainage_Density_30m : COMPLETED
Kano_Soil_Texture_30m : COMPLETED
Kano_Rainfall_30m : COMPLETED
Kano_Slope_30m : COMPLETED
Kano_Elevation_30m : COMPLETED
Kano_Elevation_30m : COMPLETED
Kano_Elevation_30m : COMPLETED
Groundwater_Potential_Kano : FAILED
Nigeria_kano : COMPLETED
twi_30m : COMPLETED
drainage_density_30m : COMPLETED
soil_texture_30m : COMPLETED
land_cover_30m : COMPLETED
slope_30m : COMPLETED
elevation_30m : COMPLETED
rainfall_30m : COMPLETED
twi_30m : COMPLETED
drainage_density_30m : COMPLETED
soil_texture_30m : COMPLETED
land_cover_30m : COM

# Export Functions
This section contains reusable helper functions for exporting Earth Engine
FeatureCollections to Google Drive. These functions are designed to minimise
code repetition and standardise exports across the project.


In [37]:
# Create an Export Function
def export_feature_collection_to_drive(
    feature_collection,
    description,
    folder,
    file_name_prefix,
    file_format="SHP"
):


    task = ee.batch.Export.table.toDrive(
        collection=feature_collection,
        description=description,
        folder=folder,
        fileNamePrefix=file_name_prefix,
        fileFormat=file_format
    )

    task.start()

    print("Export task started.")
    print("Task ID:", task.id)
    print("Description:", description)

In [ ]:
## Export AOI FeatureCollection

# The following cell exports the Area of Interest (AOI) as a Shapefile to Google Drive.
export_feature_collection_to_drive(
    feature_collection=kano,
    description="Nigeria_kano",
    folder="GEE_Exports",
    file_name_prefix="Nigeria_kano",
    file_format="SHP"
)

Export task started.
Task ID: ONRJ2X3HI6RZUDBK5NW3NACX
Description: Nigeria_kano


# CHIRPS Average Annual Precipitation (10-Year Mean)

This section loads the CHIRPS daily precipitation dataset from Google Earth Engine,
computes the average precipitation over a 10-year period, and visualises the result
using geemap.

In [38]:
#Create a new map
Map = geemap.Map(center=[9.0820, 8.6753], zoom=6)
# This map is centered on Nigeria with a zoom level of 6. You can adjust the center coordinates and zoom level as needed.
#Load the CHIRPS dataset
chirps = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate("2014-01-01", "2023-12-31"))
# Compute the Average precipitation
average_precipitation = chirps.mean()
# This computes the mean daily precipitation over the selected 10-year period.
# Visualisation parameters
vis_params = {
    "min": 0,
    "max": 20,
    "palette": [
        "white",
        "lightblue",
        "blue",
        "green",
        "yellow",
        "orange",
        "red"
    ]
}
# Display on the map
Map.addLayer(
    average_precipitation.clip(kano),
    vis_params,
    "10-Year Average Precipitation"
)
Map

Map(center=[9.082, 8.6753], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Reclassifying Continuous Layers Using Percentile Breaks

This section defines a reusable function that computes the
20th, 40th, 60th and 80th percentile values of any continuous
Earth Engine image and reclassifies it into groundwater
suitability classes ranging from 1 (least suitable) to
5 (most suitable).

In [39]:
# Compute percentile thresholds

percentiles = average_precipitation.reduceRegion(
    reducer=ee.Reducer.percentile([20, 40, 60, 80]),
    geometry=kano.geometry(),
    scale=5000,
    bestEffort=True
)

print(percentiles.getInfo())

{'precipitation_p20': 2.1608865790896945, 'precipitation_p40': 2.3829375052116286, 'precipitation_p60': 2.508218765258789, 'precipitation_p80': 2.615793144550773}


In [40]:
def reclassify_by_percentiles(image, thresholds):

    p20 = ee.Number(thresholds.get("precipitation_p20"))
    p40 = ee.Number(thresholds.get("precipitation_p40"))
    p60 = ee.Number(thresholds.get("precipitation_p60"))
    p80 = ee.Number(thresholds.get("precipitation_p80"))

    classified = (
        image.lt(p20).multiply(1)
        .add(image.gte(p20).And(image.lt(p40)).multiply(2))
        .add(image.gte(p40).And(image.lt(p60)).multiply(3))
        .add(image.gte(p60).And(image.lt(p80)).multiply(4))
        .add(image.gte(p80).multiply(5))
    )

    return classified

In [41]:
reclassified_rainfall = reclassify_by_percentiles(
    average_precipitation,
    percentiles
)

In [42]:
reclass_vis = {
    "min": 1,
    "max": 5,
    "palette": [
        "red",
        "orange",
        "yellow",
        "lightgreen",
        "darkgreen"
    ]
}

Map = geemap.Map()

Map.centerObject(kano,8)

Map.addLayer(
    reclassified_rainfall.clip(kano),
    reclass_vis,
    "Rainfall Suitability"
)

Map

Map(center=[11.735699301548124, 8.513584659731881], controls=(WidgetControl(options=['position', 'transparent_…

# Reclassification of Thematic Layers

This section reclassifies all thematic layers into a common groundwater suitability scale
ranging from 1 (least suitable) to 5 (most suitable).

The rankings are based on hydrogeological principles and are intended to standardize
all criteria before weighted overlay analysis.

Rainfall

In [43]:
rainfall_suitability = reclassified_rainfall.rename("rainfall")

Elevation

Lower elevations generally favour groundwater accumulation.

In [44]:
elevation_clip = elevation_clip.rename("elevation")

In [45]:
elevation_suitability = (
    elevation_clip.expression(
        "(b('elevation')<=200)?5"
        ":(b('elevation')<=400)?4"
        ":(b('elevation')<=700)?3"
        ":(b('elevation')<=1000)?2"
        ":1"
    )
)

Slope

Gentle slopes promote infiltration

In [46]:
slope_clip = slope_clip.rename("slope")

In [47]:
slope_suitability = (
    slope_clip.expression(
        "(b('slope')<=2)?5"
        ":(b('slope')<=5)?4"
        ":(b('slope')<=10)?3"
        ":(b('slope')<=20)?2"
        ":1"
    )
)

Drainage Density

Low drainage density is generally more favourable.

In [48]:
drainage_density_clip = drainage_density_clip.rename("drainage")

In [49]:
drainage_suitability = (
    drainage_density_clip.expression(
        "(b('drainage')<=0.002)?5"
        ":(b('drainage')<=0.004)?4"
        ":(b('drainage')<=0.006)?3"
        ":(b('drainage')<=0.008)?2"
        ":1"
    )
)

TWI

Higher TWI indicates greater moisture accumulation.

In [50]:
twi_clip = twi_clip.rename("twi")

In [51]:
twi_suitability = (
    twi_clip.expression(
        "(b('twi')<=2)?1"
        ":(b('twi')<=4)?2"
        ":(b('twi')<=6)?3"
        ":(b('twi')<=8)?4"
        ":5"
    )
)

Land Cover

Since land cover is categorical, we use remap().

For ESA WorldCover 2021:

In [52]:
land_cover_clip = land_cover_clip.rename("landcover")

In [53]:
landcover_suitability = land_cover_clip.remap(
    [10,20,30,40,50,60,70,80,90,95,100],
    [5,4,4,5,1,2,1,1,4,5,2]
).rename("landcover")

Meaning:
| Class      | Score |
| ---------- | ----- |
| Tree Cover | 5     |
| Shrubland  | 4     |
| Grassland  | 4     |
| Cropland   | 5     |
| Built-up   | 1     |
| Bare land  | 2     |
| Snow       | 1     |
| Water      | 1     |
| Wetland    | 4     |
| Mangroves  | 5     |
| Moss       | 2     |


Soil Texture

For USDA texture classes:

In [54]:
soil_suitability = soil_texture_clip.remap(
    [1,2,3,4,5,6,7,8,9,10,11,12],
    [5,5,4,4,3,3,2,2,2,1,1,1]
).rename("soil")


print(soil_texture.bandNames().getInfo())

['b0', 'b10', 'b30', 'b60', 'b100', 'b200']


Visualizing one layer

In [55]:
suitability_vis = {
    "min":1,
    "max":5,
    "palette":[
        "white",
        "orange",
        "yellow",
        "lightgreen",
        "darkgreen"
    ]
}

Map = geemap.Map(center=[12.0,8.5], zoom=8)

Map.addLayer(
    slope_suitability,
    suitability_vis,
    "Slope Suitability"
)

Map

Map(center=[12.0, 8.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

In [56]:
layers = {
    "Rainfall": rainfall_suitability,
    "Elevation": elevation_suitability,
    "Slope": slope_suitability,
    "Landcover": landcover_suitability,
    "Soil": soil_suitability,
    "Drainage": drainage_suitability,
    "TWI": twi_suitability
}

for name, img in layers.items():
    try:
        print(f"{name}: {img.bandNames().getInfo()}")
    except Exception as e:
        print(f"{name}: ERROR")
        print(e)

Rainfall: ['rainfall']
Elevation: ['constant']
Slope: ['constant']
Landcover: ['landcover']
Soil: ['soil']
Drainage: ['constant']
TWI: ['constant']


In [57]:
print(rainfall_suitability.bandNames().getInfo())
print(elevation_suitability.bandNames().getInfo())
print(slope_suitability.bandNames().getInfo())
print(landcover_suitability.bandNames().getInfo())
print(soil_suitability.bandNames().getInfo())
print(drainage_suitability.bandNames().getInfo())
print(twi_suitability.bandNames().getInfo())

['rainfall']
['constant']
['constant']
['landcover']
['soil']
['constant']
['constant']


## Weight Overlay Performance 

In [58]:

weights = {
    "rainfall": 0.04,
    "elevation": 0.08,
    "slope": 0.03,
    "landcover": 0.07,
    "soil": 0.12,
    "drainage": 0.03,
    "twi": 0.09
}

groundwater_index = (
    rainfall_suitability.multiply(weights["rainfall"])
    .add(elevation_suitability.multiply(weights["elevation"]))
    .add(slope_suitability.multiply(weights["slope"]))
    .add(landcover_suitability.multiply(weights["landcover"]))
    .add(soil_suitability.multiply(weights["soil"]))
    .add(drainage_suitability.multiply(weights["drainage"]))
    .add(twi_suitability.multiply(weights["twi"]))
).rename("Groundwater_Index")

## Classify into five groundwater potential classes

In [59]:
groundwater_classes = (
    groundwater_index
    .lt(0.3).multiply(1)
    .add(groundwater_index.gte(0.3).And(groundwater_index.lt(0.6)).multiply(2))
    .add(groundwater_index.gte(0.6).And(groundwater_index.lt(0.9)).multiply(3))
    .add(groundwater_index.gte(0.9).And(groundwater_index.lt(1.2)).multiply(4))
    .add(groundwater_index.gte(1.2).multiply(5))
)

In [60]:
groundwater_index = (
    rainfall_suitability.multiply(weights["rainfall"])
    .add(elevation_suitability.multiply(weights["elevation"]))
    .add(slope_suitability.multiply(weights["slope"]))
    .add(landcover_suitability.multiply(weights["landcover"]))
    .add(soil_suitability.multiply(weights["soil"]))
    .add(drainage_suitability.multiply(weights["drainage"]))
    .add(twi_suitability.multiply(weights["twi"]))
)

print(groundwater_index.bandNames().getInfo())
print(type(groundwater_index))


['rainfall']
<class 'ee.image.Image'>


Visualize the classified map

In [61]:
class_vis = {
    "min": 1,
    "max": 5,
    "palette": [
        "red",
        "orange",
        "yellow",
        "lightgreen",
        "darkgreen"
    ]
}

Map = geemap.Map(center=[12.0, 8.5], zoom=8)

Map.addLayer(
    groundwater_classes,
    class_vis,
    "Groundwater Potential Classes"
)
legend = {
    "Very Low": "red",
    "Low": "orange",
    "Moderate": "yellow",
    "High": "lightgreen",
    "Very High": "darkgreen"
}

Map.add_legend(
    title="Groundwater Potential",
    legend_dict=legend
)

Map
Map

Map(center=[12.0, 8.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

| Raster Value | Groundwater Potential |
| ------------ | --------------------- |
| 1            | Very Low              |
| 2            | Low                   |
| 3            | Moderate              |
| 4            | High                  |
| 5            | Very High             |


Convert the classified raster to polygons

In [62]:
groundwater_vector = groundwater_classes.reduceToVectors(
    geometry=kano.geometry(),
    geometryType="polygon",
    reducer=ee.Reducer.countEvery(),
    scale=30,
    maxPixels=1e13
)

Compute the area of each class (km²)

In [64]:
groundwater_vector = groundwater_vector.map(
    lambda feature: feature.set(
        "Area_km2",
        feature.geometry().area().divide(1e6)
    )
)

print(groundwater_vector.limit(10).getInfo())

{'type': 'FeatureCollection', 'columns': {'Area_km2': 'Number', 'count': 'Long<0, 4294967295>', 'label': 'Byte<0, 15>', 'system:index': 'String'}, 'features': []}


In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=groundwater_vector,
    description="Groundwater_Potential_Kano",
    folder="GEE_Exports",
    fileFormat="SHP"
)

task.start()

print("Export started...")

Export started...


In [ ]:
import ee

ee.Initialize()

task = ee.batch.Export.image.toDrive(
    image=land_cover,
    description="Kano_Land_Cover_30m",
    folder="Groundwater_Kano",
    fileNamePrefix="Kano_Land_Cover_30m",
    region=kano.geometry(),
    scale=30,
    crs="EPSG:4326",
    maxPixels=1e13
)

task.start()

print("Export started:", task.id)

Export started: DGYMR4GFQC3IPWEQSHNUZGT5


In [65]:
# Check that all seven suitability layers exist

reclassified_layers = {
    "Rainfall": rainfall_suitability,
    "Elevation": elevation_suitability,
    "Slope": slope_suitability,
    "Landcover": landcover_suitability,
    "Soil": soil_suitability,
    "Drainage": drainage_suitability,
    "TWI": twi_suitability
}

for name, image in reclassified_layers.items():

    try:
        print(
            f"{name}: "
            f"{image.bandNames().getInfo()}"
        )

    except Exception as e:
        print(f"{name}: ERROR")
        print(e)

Rainfall: ['rainfall']
Elevation: ['constant']
Slope: ['constant']
Landcover: ['landcover']
Soil: ['soil']
Drainage: ['constant']
TWI: ['constant']


In [66]:
# ------------------------------------------------------------
# 1. Create dictionary of the reclassified/suitability layers
# ------------------------------------------------------------

reclassified_layers = {
    "Kano_Reclassified_Rainfall": rainfall_suitability,
    "Kano_Reclassified_Elevation": elevation_suitability,
    "Kano_Reclassified_Slope": slope_suitability,
    "Kano_Reclassified_Landcover": landcover_suitability,
    "Kano_Reclassified_Soil": soil_suitability,
    "Kano_Reclassified_Drainage": drainage_suitability,
    "Kano_Reclassified_TWI": twi_suitability
}

# ------------------------------------------------------------
# 2. Set export region
# ------------------------------------------------------------

region = kano.geometry()

# ------------------------------------------------------------
# 3. Google Drive folder
# ------------------------------------------------------------

drive_folder = "Groundwater_Kano_Reclassified1"

# ------------------------------------------------------------
# 4. Start exports
# ------------------------------------------------------------

tasks = []

for name, image in reclassified_layers.items():

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=name,
        folder=drive_folder,
        fileNamePrefix=name,
        region=region,
        scale=30,
        crs="EPSG:4326",
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )

    task.start()
    tasks.append(task)

    print(f"Started export: {name}")

print("\nAll reclassified export tasks have been started.")

Started export: Kano_Reclassified_Rainfall
Started export: Kano_Reclassified_Elevation
Started export: Kano_Reclassified_Slope
Started export: Kano_Reclassified_Landcover
Started export: Kano_Reclassified_Soil
Started export: Kano_Reclassified_Drainage
Started export: Kano_Reclassified_TWI

All reclassified export tasks have been started.


In [67]:

ee.Initialize()

task = ee.batch.Export.image.toDrive(
    image=groundwater_classes.rename("Groundwater_Potential"),
    description="Kano_Groundwater_Potential",
    folder="Groundwater_Kano_Final",
    fileNamePrefix="Kano_Groundwater_Potential",
    region=kano.geometry(),
    scale=30,
    crs="EPSG:4326",
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()

print("Groundwater export started.")
print("Task ID:", task.id)

Groundwater export started.
Task ID: YTI2JQSZVRXXWKNY4KOGTYKI
